In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.inspection import permutation_importance
import matplotlib

In [3]:
df = pd.read_csv("concrete_compressive_strength.csv")
print(df.head())

   Cement   Slag  FlyAsh  Water  Superplasticizer  CoarseAgg  FineAgg  Age  \
0   540.0    0.0     0.0  162.0               2.5     1040.0    676.0   28   
1   540.0    0.0     0.0  162.0               2.5     1055.0    676.0   28   
2   332.5  142.5     0.0  228.0               0.0      932.0    594.0  270   
3   332.5  142.5     0.0  228.0               0.0      932.0    594.0  365   
4   198.6  132.4     0.0  192.0               0.0      978.4    825.5  360   

    Strength  
0  79.986111  
1  61.887366  
2  40.269535  
3  41.052780  
4  44.296075  


In [4]:
X = df.drop('Strength', axis=1)
y = df['Strength']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)   
X_test_sc  = scaler.transform(X_test)        

In [7]:
mlr = LinearRegression()
mlr.fit(X_train_sc, y_train)
y_pred_mlr = mlr.predict(X_test_sc)

coef_df = pd.DataFrame({
    'Feature'    : df.columns[:-1],
    'Coefficient': mlr.coef_
}).sort_values('Coefficient', ascending=False)

print(f"\nIntercept : {mlr.intercept_:.4f}")
print(f"\nCoefficients (on scaled features):")
print(coef_df.to_string(index=False))
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train_sc, y_train)
y_pred_rf = rf.predict(X_test_sc)
fi = pd.Series(rf.feature_importances_, index=df.columns[:-1]).sort_values(ascending=False)
print(f"\nFeature Importance (MDI):")
print(fi.round(4))


Intercept : 35.8577

Coefficients (on scaled features):
         Feature  Coefficient
          Cement    12.786504
            Slag     9.432883
             Age     7.037787
          FlyAsh     5.255609
         FineAgg     1.947380
Superplasticizer     1.841103
       CoarseAgg     1.400255
           Water    -2.892085

Feature Importance (MDI):
Age                 0.3328
Cement              0.3245
Water               0.1240
Slag                0.0762
Superplasticizer    0.0589
FineAgg             0.0358
CoarseAgg           0.0280
FlyAsh              0.0198
dtype: float64


In [8]:
def evaluate(y_true, y_pred, name):
    mse  = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - y_true.mean()) ** 2)
    adj_r2 = 1 - (ss_res / (len(y_true) - 8 - 1)) / (ss_tot / (len(y_true) - 1))
    mask   = y_true.values != 0
    mape   = np.mean(np.abs((y_true.values[mask] - y_pred[mask])
                             / y_true.values[mask])) * 100
    return {
        'Model'   : name,
        'MSE'     : mse,
        'RMSE'    : rmse,
        'MAE'     : mae,
        'R2'      : r2,
        'Adj_R2'  : adj_r2,
        'MAPE(%)'  : mape
    }

metrics = [
    evaluate(y_test, y_pred_mlr, 'Multiple LR'),
    evaluate(y_test, y_pred_rf,  'Random Forest'),
]
result_df = pd.DataFrame(metrics).set_index('Model')
print(result_df.round(4).to_string())

print("\n[Metric Guide]")
print("  MSE     : Mean Squared Error  (unit: MPa²) — large errors penalized more")
print("  RMSE    : Root MSE            (unit: MPa)  — same scale as target")
print("  MAE     : Mean Absolute Error (unit: MPa)  — robust to outliers")
print("  R²      : Variance explained  (0~1)        — 1 = perfect")
print("  Adj R²  : R² penalized for # of features")
print("  MAPE(%) : % error — scale-independent, intuitive for non-experts")

                   MSE    RMSE     MAE      R2  Adj_R2  MAPE(%)
Model                                                          
Multiple LR    95.9755  9.7967  7.7454  0.6275  0.6124  29.2763
Random Forest  30.8463  5.5539  3.7813  0.8803  0.8754  12.3793

[Metric Guide]
  MSE     : Mean Squared Error  (unit: MPa²) — large errors penalized more
  RMSE    : Root MSE            (unit: MPa)  — same scale as target
  MAE     : Mean Absolute Error (unit: MPa)  — robust to outliers
  R²      : Variance explained  (0~1)        — 1 = perfect
  Adj R²  : R² penalized for # of features
  MAPE(%) : % error — scale-independent, intuitive for non-experts
